# BERT Family QA (Extractive Question Answering) 教學範例

## 教學目標

- 了解 Extractive QA 的任務定義：從 context 中找出答案的起始與結束位置
- 理解 offset mapping 如何將「字元級別」的答案標註轉換成「token 級別」的標註
- 使用 Hugging Face Transformers 完成 SQuAD 資料集上的 fine-tuning

## 教學內容

- 使用 DistilBERT 作為 pre-trained model，在 [SQuAD 資料集](https://huggingface.co/datasets/rajpurkar/squad)的子集上進行 fine-tuning
- 這是一個 **Extractive QA** 任務：模型不會自己「生成」答案，而是從 context 中「抽取」一段連續的文字作為答案

In [ ]:
# 0. 引入相關套件

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DefaultDataCollator,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

In [ ]:
# 1. 超參數區

MODEL_NAME = "distilbert/distilbert-base-uncased"
LR = 2e-5
TRAIN_BATCH_SIZE = 16  # 訓練時每個 batch 的樣本數
EVAL_BATCH_SIZE = 16   # 評估時每個 batch 的樣本數
NUM_TRAIN_EPOCHS = 3

## 載入資料集

[SQuAD (Stanford Question Answering Dataset)](https://rajpurkar.github.io/SQuAD-explorer/) 是最經典的 Extractive QA 資料集。每筆資料包含：

| 欄位 | 說明 | 範例 |
| - | - | - |
| `question` | 問題 | "What is gravity?" |
| `context` | 包含答案的段落 | "Gravity is a force..." |
| `answers` | 答案文字 + 在 context 中的字元位置 | `{"text": ["a force"], "answer_start": [11]}` |

為了加速教學示範，我們只取前 5000 筆資料。

In [ ]:
# 2. 只載入前 5000 筆資料做示範（完整訓練集有 87,599 筆）

# 使用 datasets 庫載入 SQuAD (Stanford Question Answering Dataset)
# split="train[:5000]" 代表僅抽取訓練集的前 5,000 筆樣本，以加快教學演練的速度
# SQuAD 的每筆資料包含：id, title, context (文章), question (問題) 以及 answers (包含答案文字與起始字元索引)
squad = load_dataset("squad", split="train[:5000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# 3. 切分訓練集與測試集
# 原始的 SQuAD 有自己的 validation split，但因為我們只取了一小部分，所以自己切 80/20。

squad = squad.train_test_split(test_size=0.2)

In [ ]:
# 4. 觀察資料結構
# 取出 test split 的前 10 筆資料
# 注意：這裡的 examples 是 dict of lists

examples = squad["test"][0:10]
print(f"資料裡面有什麼: {examples.keys()}")
print(f"第 0 筆內容: {examples['context'][0]}")
print(f"第 0 筆問題: {examples['question'][0]}")
print(f"第 0 筆答案: {examples['answers'][0]}")

資料裡面有什麼: dict_keys(['id', 'title', 'context', 'question', 'answers'])
第 0 筆內容: Tracks of the Northern Pacific Railroad (NPR) reached Montana from the west in 1881 and from the east in 1882. However, the railroad played a major role in sparking tensions with Native American tribes in the 1870s. Jay Cooke, the NPR president launched major surveys into the Yellowstone valley in 1871, 1872 and 1873 which were challenged forcefully by the Sioux under chief Sitting Bull. These clashes, in part, contributed to the Panic of 1873 which delayed construction of the railroad into Montana. Surveys in 1874, 1875 and 1876 helped spark the Great Sioux War of 1876. The transcontinental NPR was completed on September 8, 1883, at Gold Creek.
第 0 筆問題: When did the Northern Pacific Railroad reach Montana from the east?
第 0 筆答案: {'text': ['1882'], 'answer_start': [105]}


## Tokenizer
Recap: Tokenizer 是負責把原始文字轉成模型能讀的 token IDs 的工具。

In [ ]:
# 5. 載入 Tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### Tokenizer 的輸入與輸出

QA 任務的 tokenizer 需要同時接收 `question` 和 `context` 兩段文字。

輸入格式會變成：`[CLS] question tokens [SEP] context tokens [SEP]`

幾個重要的參數：

| 參數 | 說明 |
| - | - |
| `max_length=384` | token 序列的最大長度 |
| `truncation="only_second"` | 如果太長，只截斷 context（不截斷 question） |
| `return_offsets_mapping=True` | 回傳每個 token 對應的原文字元位置 |
| `padding="max_length"` | 不足 384 的部分補 padding |

In [ ]:
# 6. 觀察 Tokenizer 的輸入與輸出

questions = [q.strip() for q in examples["question"]]

inputs = tokenizer(
    questions,                        # 第一個句子：question
    examples["context"],              # 第二個句子：context
    max_length=384,
    truncation="only_second",          # 只截斷 context
    return_offsets_mapping=True,       # 回傳 offset mapping
    padding="max_length",
)

print(f"第0筆資料的 input_ids: {inputs['input_ids'][0]}")
print(f"第0筆資料的 attention_mask: {inputs['attention_mask'][0]}")
print(f"第0筆資料的 token_type_ids: {inputs['token_type_ids'][0]}")
print(f"第0筆資料的 offset_mapping: {inputs['offset_mapping'][0]}")

第0筆資料的 input_ids: [101, 2043, 2106, 1996, 2642, 3534, 4296, 3362, 8124, 2013, 1996, 2264, 1029, 102, 3162, 1997, 1996, 2642, 3534, 4296, 1006, 21411, 1007, 2584, 8124, 2013, 1996, 2225, 1999, 7005, 1998, 2013, 1996, 2264, 1999, 7131, 1012, 2174, 1010, 1996, 4296, 2209, 1037, 2350, 2535, 1999, 12125, 2075, 13136, 2007, 3128, 2137, 6946, 1999, 1996, 14896, 1012, 6108, 16546, 1010, 1996, 21411, 2343, 3390, 2350, 12265, 2046, 1996, 29231, 3028, 1999, 7428, 1010, 7572, 1998, 7612, 2029, 2020, 8315, 23097, 2011, 1996, 16615, 2104, 2708, 3564, 7087, 1012, 2122, 17783, 1010, 1999, 2112, 1010, 5201, 2000, 1996, 6634, 1997, 7612, 2029, 8394, 2810, 1997, 1996, 4296, 2046, 8124, 1012, 12265, 1999, 7586, 1010, 7466, 1998, 7326, 3271, 12125, 1996, 2307, 16615, 2162, 1997, 7326, 1012, 1996, 9099, 8663, 10196, 15758, 21411, 2001, 2949, 2006, 2244, 1022, 1010, 7257, 1010, 2012, 2751, 3636, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
# 7. 觀察 sequence_ids
# sequence_ids 是 tokenizer 提供的方法，用來標記每個 token 屬於哪個輸入句子。

# 印出這個batch中第 0 筆資料的 sequence_ids
inputs.sequence_ids(0)

[None,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 None,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 N

## Offset Mapping 是什麼？——以 BERT QA 為例

### 核心問題

Tokenizer 會把原始文字切成 **subword tokens**，切完之後，每個 token 對應到原文的「哪幾個字元」就不再明顯了。**Offset mapping 就是一張對照表，記錄每個 token 在原始字串中的起始與結束字元位置。**

---

### 具體範例

假設原始句子是：

```
"Transformers are amazing"
```

Tokenizer 可能切成：

| Token Index | Token        | Offset (start, end) | 對應原文字元          |
|:-----------:|:-------------|:-------------------:|:---------------------|
| 0           | `[CLS]`      | (0, 0)              | *(特殊 token，無對應)* |
| 1           | `Transform`  | (0, 9)              | `Transform`          |
| 2           | `##ers`      | (9, 12)             | `ers`                |
| 3           | `are`        | (13, 16)            | `are`                |
| 4           | `amazing`    | (17, 24)            | `amazing`            |
| 5           | `[SEP]`      | (0, 0)              | *(特殊 token，無對應)* |

重點觀察：

- `"Transformers"` 被拆成兩個 token：`Transform` (字元 0–9) 和 `##ers` (字元 9–12)。
- 特殊 token `[CLS]`、`[SEP]` 的 offset 是 `(0, 0)`，代表它們不對應原文中的任何字元。
- offset 使用的是 **左閉右開** 區間，也就是 `原文[start:end]` 就能取出該 token 對應的原始文字。

---

### 為什麼 QA 任務特別需要它？

在 QA 任務中，訓練資料的答案標註方式是：

```json
{
  "text": ["amazing"],
  "answer_start": [17]
}
```

這裡的 `17` 是 **字元級別 (character-level)** 的位置，但模型預測的是 **token 級別 (token-level)** 的位置。

所以我們需要做一個轉換：

```
字元位置 (answer_start=17) ──→ token 位置 (token_index=4)
        ↑
   offset mapping 就是這座橋
```

### 一句話總結

> **Offset mapping 是 tokenizer 提供的「token ↔ 原始字元位置」對照表，讓我們能把人類標註的字元級答案位置，精確轉換成模型需要的 token 級位置。**

## 前處理函式 `preprocess_function`

這是整個 QA 前處理最核心的部分。目標是把人類標註的「字元級別答案位置」轉換成「token 級別答案位置」。

因為會搭配 `dataset.map(batched=True)` 使用，所以 `examples` 是一整個 batch（dict of lists），不是單筆資料。

### 整體流程

```
[輸入] examples（一個 batch 的 question + context + answers）
  │
  ├─ 1. Tokenize：把 question 和 context 一起編碼
  ├─ 2. 取出 offset_mapping（token ↔ 字元位置的對照表）
  ├─ 3. 對每筆資料：
  │     ├─ 第一步：算出答案的字元範圍 (start_char, end_char)
  │     ├─ 第二步：用 sequence_ids 找到 context 的 token 範圍
  │     ├─ 第三步：檢查答案是否在 context 範圍內
  │     └─ 第四步：用 offset_mapping 把字元位置轉成 token 位置
  │
  └─ [輸出] inputs + start_positions + end_positions
```

In [ ]:
# 8. 定義前處理函式

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    # 取出 offset_mapping 後從 inputs 中移除（模型不需要它）
    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]

        # === 第一步：取得答案的字元範圍 ===
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])

        # === 第二步：用 sequence_ids 找到 context 的 token 範圍 ===
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the `context`
        # 理論上答案只能在 `context` 中
        # `idx` 代表 token 在序列中的位置
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # === 第三步：檢查答案是否完整落在 context 範圍內 ===
        # 如果答案被 truncation 截掉了，標記為 (0, 0)

        # offsets 的第一個的開頭已經比答案的終點 (`end_char`) 還大
        # 或 offsets 的最後一個的結尾已經比答案的起點 (`start_char`) 還小
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # === 第四步：用 offset mapping 把字元位置轉成 token 位置 ===

            # 找 start token：從 context 開頭往右掃，找到最後一個「起始字元 ≤ start_char」的 token
            idx = context_start
            # [條件1] idx <= context_end : 代表現在要遍歷的 token 位置不能超過 context
            # [條件2] offset[idx][0] <= start_char : 代表左邊位置比 start_char 還小
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            # 找 end token：從 context 結尾往左掃，找到最後一個「結束字元 ≥ end_char」的 token
            idx = context_end
            # [條件1] idx <= context_start : 代表現在要遍歷的 token 位置已在 context 開頭的右側
            # [條件2] offset[idx][1] >= end_char : 代表右邊位置比 end_char 還大
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions # batch 形式
    inputs["end_positions"] = end_positions # batch 形式
    return inputs

### `preprocess_function` 補充說明

整個 `preprocess_function` 的流程可以拆成四步：

#### 第一步：取得答案的字元範圍

```python
start_char = answer["answer_start"][0]                        # 答案起始字元位置
end_char   = answer["answer_start"][0] + len(answer["text"][0])  # 答案結束字元位置
```

以上面的例子：`start_char = 17`, `end_char = 24`。

#### 第二步：找到 context 在 token 序列中的範圍

因為輸入格式是 `[CLS] question [SEP] context [SEP]`，需要用 `sequence_ids()` 找出 context 對應的 token 區段（`sequence_ids == 1` 的部分）。

```
sequence_ids:  [None, 0, 0, 0, None, 1, 1, 1, 1, None]
                 ↑   question   ↑     context      ↑
               [CLS]          [SEP]               [SEP]
```

#### 第三步：檢查答案是否在 context 範圍內

利用 offset mapping，比對答案的字元範圍與 context token 的字元範圍是否重疊。如果答案不在範圍內（例如被 truncation 截掉了），就標記為 `(0, 0)`。

#### 第四步：用 offset mapping 找到答案的 token 位置

從 context 的起點開始，逐一比對每個 token 的 offset，找到：

- **start token**：第一個 offset 起點 `≤ start_char` 的最後一個 token
- **end token**：最後一個 offset 終點 `≥ end_char` 的第一個 token

---

In [ ]:
# 9. 對整個資料集執行前處理

# 使用 `.map()` 搭配 `batched=True`，一次處理多筆資料
tokenized_squad = squad.map(
    preprocess_function,
    batched=True,
    remove_columns=squad["train"].column_names,
)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# 10. 準備 Data Collator (負責把多筆資料整理成一個 batch 的 tensor)

# 因為我們在前處理時已經做好 padding 了（`padding="max_length"`），所以用最基本的 collator 即可
data_collator = DefaultDataCollator()

In [ ]:
# 11. 載入 Pre-trained Model

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# 12. 設定訓練參數並開始訓練

training_args = TrainingArguments(
    output_dir="my_awesome_qa_model",
    eval_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.009271
2,2.601463,1.618198
3,2.601463,1.564477


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=2.1887428385416667, metrics={'train_runtime': 532.6679, 'train_samples_per_second': 22.528, 'train_steps_per_second': 1.408, 'total_flos': 1175877900288000.0, 'train_loss': 2.1887428385416667, 'epoch': 3.0})

## 結語
- 以上是我們對於 QA 的任務的實作介紹。
- 若要進行模型後續評估，請參見：https://huggingface.co/learn/llm-course/chapter7/7?fw=pt#post-processing